# 02_02：依 PERMNO 人工審核清理 comment–ticker–PERMNO

讀取 pipeline registry 的新版 02 與同 run 的 `top300_permno_review.csv`。僅排除 del=1 的 PERMNO，del=0 及未審核 PERMNO 保留；不再使用舊 ticker 審核表或固定舊來源。

核對審核表的來源 SHA-256 與每個 PERMNO 的 times。完整保留來源欄位、日期、列順序與 PERMNO，不重新配對。每次建立新的輸出目錄，原始資料與人工 del 不改動。unique comments 仍以 `(source_file, source_row)` 識別。


In [4]:
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import csv, json, hashlib, sqlite3, time
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq
import pandas as pd
from IPython.display import display

ROOT=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'Data/process').is_dir())
MATCH_RUN = None  # 可指定新版 02 run；預設使用成功 registry。
if MATCH_RUN is None:
    pointer = json.loads((ROOT/'Data/process/pipeline_runs.json').read_text(encoding='utf-8')).get('matching_manifest')
    if not pointer: raise FileNotFoundError('請先成功執行新版 02。')
    manifest = Path(pointer)
    if not manifest.is_absolute(): manifest = ROOT/manifest
    MATCH_RUN = manifest.parent
else:
    MATCH_RUN = Path(MATCH_RUN)
    if not MATCH_RUN.is_absolute(): MATCH_RUN = ROOT/MATCH_RUN
matching_info = json.loads((MATCH_RUN/'run_summary.json').read_text(encoding='utf-8'))
assert matching_info['status']=='complete' and matching_info.get('schema_version',0)>=3
SOURCE = MATCH_RUN/'comment_ticker_long.parquet'
assert 'permno' in pq.read_schema(SOURCE).names, '來源缺少已按留言日期配對的 PERMNO。'
REVIEW = ROOT/'Data/process/02_01_permno_frequency_review'/MATCH_RUN.name/'top300_permno_review.csv'
review_bytes = REVIEW.read_bytes()
review = pd.read_csv(REVIEW,dtype=str,keep_default_na=False,encoding='utf-8-sig')
assert review.columns.tolist()==['permno','ticker','times','del']
assert review.permno.str.fullmatch(r'[1-9][0-9]*').all() and review.permno.is_unique
assert review.times.str.fullmatch(r'[1-9][0-9]*').all()
assert review['del'].isin(['0','1']).all(), 'del 僅接受 0 或 1。'
excluded = set(map(int,review.loc[review['del'].eq('1'),'permno']))
OUT=ROOT/'Data/process/02_02_filtered_comment_ticker'/datetime.now(timezone.utc).strftime('%Y-%m-%d_%H-%M-%S_UTC')
OUT.mkdir(parents=True,exist_ok=False)
DEST=OUT/'comment_ticker_cleaned.parquet'
TEMP=DEST.with_suffix('.parquet.partial')
source_stat=(SOURCE.stat().st_size,SOURCE.stat().st_mtime_ns)
def sha256_file(path):
    digest=hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda:stream.read(4*1024*1024),b''):digest.update(block)
    return digest.hexdigest()
source_hash=sha256_file(SOURCE)
review_meta_path=REVIEW.parent/'run_summary.json'
if review_meta_path.exists():
    review_meta=json.loads(review_meta_path.read_text(encoding='utf-8'))
    assert Path(review_meta['source']).resolve()==SOURCE.resolve()
    assert review_meta['source_sha256']==source_hash, '人工審核來源與目前 long-format 檔案不一致'
print('來源：',SOURCE)
print('審核表列數：',len(review),'；del=1 PERMNO：',len(excluded))
print('输出：',DEST)


來源： c:\Users\eagle\RedditStock\Data\process\02_ticker_matching\2026-10-15_02-43-43_UTC\comment_ticker_long.parquet
審核表列數： 300 ；del=1 PERMNO： 10
输出： c:\Users\eagle\RedditStock\Data\process\02_02_filtered_comment_ticker\2026-10-15_05-58-53_UTC\comment_ticker_cleaned.parquet


## 分批過濾與診斷
只移除排除集合中的 PERMNO，保留完整欄位。


In [5]:
started=time.perf_counter()
parquet=pq.ParquetFile(SOURCE)
assert {'permno','ticker','year_month','thread_id','comment_id','source_file','source_row'}<=set(parquet.schema_arrow.names)
deny=pa.array(sorted(excluded),type=pa.int64())
db_path=OUT/'_diagnostic_work.sqlite'
db=sqlite3.connect(db_path)
db.execute('PRAGMA temp_store=FILE')
db.execute('PRAGMA cache_size=-32768')
db.execute('CREATE TABLE comments(stage TEXT, file TEXT, row INTEGER, PRIMARY KEY(stage,file,row)) WITHOUT ROWID')
db.execute('CREATE TABLE threads(stage TEXT, thread TEXT, PRIMARY KEY(stage,thread)) WITHOUT ROWID')
stats={stage:{'rows':0,'month_min':None,'month_max':None,'missing_month_rows':0,'tickers':set(),'permnos':set()} for stage in ['before','after']}
observed=Counter()
def update(stage,table):
    s=stats[stage];s['rows']+=table.num_rows
    s['tickers'].update(pc.unique(table.column('ticker')).to_pylist())
    s['permnos'].update(pc.unique(table.column('permno')).to_pylist())
    months=[x for x in pc.unique(table.column('year_month')).to_pylist() if x]
    if months:
        s['month_min']=min(s['month_min'],min(months)) if s['month_min'] else min(months)
        s['month_max']=max(s['month_max'],max(months)) if s['month_max'] else max(months)
    missing=pc.or_(pc.is_null(table.column('year_month')),pc.fill_null(pc.equal(table.column('year_month'),''),False))
    s['missing_month_rows']+=int(pc.sum(pc.cast(missing,pa.int64())).as_py() or 0)
    comments=table.select(['source_file','source_row']).to_pandas().drop_duplicates()
    assert comments.notna().all().all()
    db.executemany('INSERT OR IGNORE INTO comments VALUES (?,?,?)',((stage,f,int(r)) for f,r in comments.itertuples(index=False,name=None)))
    db.executemany('INSERT OR IGNORE INTO threads VALUES (?,?)',((stage,t) for t in pc.unique(table.column('thread_id')).to_pylist() if t))
    return {'row_count':table.num_rows,'month_min':min(months) if months else None,'month_max':max(months) if months else None,
            'unique_comments':len(comments),'unique_threads':len([t for t in pc.unique(table.column('thread_id')).to_pylist() if t]),'unique_authors':'N/A'}
with pq.ParquetWriter(TEMP,parquet.schema_arrow,compression='zstd') as writer:
    for batch_no,batch in enumerate(parquet.iter_batches(batch_size=100_000),1):
        table=pa.Table.from_batches([batch])
        assert table.column('ticker').null_count==0
        assert table.column('permno').null_count==0
        assert pc.all(pc.greater(table.column('permno'),0)).as_py()
        for item in pc.value_counts(table.column('permno')).to_pylist():observed[item['values']]+=item['counts']
        cleaned=table.filter(pc.invert(pc.is_in(table.column('permno'),value_set=deny)))
        assert not any(t in excluded for t in pc.unique(cleaned.column('permno')).to_pylist())
        before_diag=update('before',table);after_diag=update('after',cleaned)
        if cleaned.num_rows:writer.write_table(cleaned,row_group_size=100_000)
        db.commit()
        print({'batch':batch_no,'before':before_diag,'after':after_diag},flush=True)
assert stats['before']['rows']==parquet.metadata.num_rows
assert pq.read_metadata(TEMP).num_rows==stats['after']['rows']
# 比對審核表上的 times，避免同名 run 或來源混用。
for row in review.itertuples(index=False,name=None):
    assert observed[int(row[0])]==int(row[2]), ('審核表 times 與來源不同',row[0])
assert stats['before']['rows']-stats['after']['rows']==sum(observed[t] for t in excluded)
for stage,s in stats.items():
    s['unique_tickers']=len(s.pop('tickers'))
    s['unique_permnos']=len(s.pop('permnos'))
    s['unique_comments']=db.execute('SELECT COUNT(*) FROM comments WHERE stage=?',(stage,)).fetchone()[0]
    s['unique_threads']=db.execute('SELECT COUNT(*) FROM threads WHERE stage=?',(stage,)).fetchone()[0]
    s['unique_authors']=None
db.close()
assert REVIEW.read_bytes()==review_bytes
assert source_stat==(SOURCE.stat().st_size,SOURCE.stat().st_mtime_ns)
TEMP.replace(DEST)
db_path.unlink()
report={'status':'complete','source':str(SOURCE),'source_sha256':source_hash,'review_file':str(REVIEW),
        'review_sha256':hashlib.sha256(review_bytes).hexdigest(),'review_rows':len(review),
        'schema_version':2,'review_unit':'permno','excluded_permnos':sorted(excluded),'excluded_permno_count':len(excluded),
        'before':stats['before'],'after':stats['after'],
        'removed_mentions':stats['before']['rows']-stats['after']['rows'],
        'comments_losing_all_tickers':stats['before']['unique_comments']-stats['after']['unique_comments'],
        'output':str(DEST),'processing_seconds':time.perf_counter()-started}
(OUT/'run_summary.json').write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf-8')
display(pd.DataFrame(stats).T)
preview=next(pq.ParquetFile(DEST).iter_batches(batch_size=10),None)
if preview is not None:display(preview.to_pandas())
print('完成：',DEST)


{'batch': 1, 'before': {'row_count': 100000, 'month_min': '2012-05', 'month_max': '2015-08', 'unique_comments': 46499, 'unique_threads': 6322, 'unique_authors': 'N/A'}, 'after': {'row_count': 52641, 'month_min': '2012-05', 'month_max': '2015-08', 'unique_comments': 31666, 'unique_threads': 5872, 'unique_authors': 'N/A'}}
{'batch': 2, 'before': {'row_count': 100000, 'month_min': '2015-08', 'month_max': '2015-12', 'unique_comments': 49025, 'unique_threads': 5342, 'unique_authors': 'N/A'}, 'after': {'row_count': 52116, 'month_min': '2015-08', 'month_max': '2015-12', 'unique_comments': 33079, 'unique_threads': 4938, 'unique_authors': 'N/A'}}
{'batch': 3, 'before': {'row_count': 100000, 'month_min': '2015-12', 'month_max': '2016-04', 'unique_comments': 50383, 'unique_threads': 5242, 'unique_authors': 'N/A'}, 'after': {'row_count': 50912, 'month_min': '2015-12', 'month_max': '2016-04', 'unique_comments': 33279, 'unique_threads': 4815, 'unique_authors': 'N/A'}}
{'batch': 4, 'before': {'row_co

,rows,month_min,month_max,missing_month_rows,unique_tickers,unique_permnos,unique_comments,unique_threads,unique_authors
before,10049767,2012-05,2020-07,0,3936,3767,5951443,241473,None
after,4994423,2012-05,2020-07,0,3927,3757,3742825,221914,None


,year_month,thread_id,comment_id,ticker,source_file,source_row,permno,created_date,created_datetime_utc,match_status
0,2012-05,t0olb,c4imwtq,AWAY,wallstreetbets_comment_part1.csv,2,12882,2012-05-01,2012-05-01 05:07:58+00:00,matched
1,2012-05,t0olb,c4imwtq,DVD,wallstreetbets_comment_part1.csv,2,84026,2012-05-01,2012-05-01 05:07:58+00:00,matched
2,2012-05,t0olb,c4imwtq,LOOK,wallstreetbets_comment_part1.csv,2,87173,2012-05-01,2012-05-01 05:07:58+00:00,matched
3,2012-05,t0olb,c4imwtq,LOW,wallstreetbets_comment_part1.csv,2,61399,2012-05-01,2012-05-01 05:07:58+00:00,matched
4,2012-05,t0olb,c4imwtq,POST,wallstreetbets_comment_part1.csv,2,13188,2012-05-01,2012-05-01 05:07:58+00:00,matched
5,2012-05,suslt,c4iq6sd,OPEN,wallstreetbets_comment_part1.csv,3,92934,2012-05-01,2012-05-01 13:36:07+00:00,matched
6,2012-05,t0olb,c4iqjuz,NFLX,wallstreetbets_comment_part1.csv,4,89393,2012-05-01,2012-05-01 14:22:35+00:00,matched
7,2012-05,svadz,c4ird5f,GET,wallstreetbets_comment_part1.csv,5,85426,2012-05-01,2012-05-01 15:44:24+00:00,matched
8,2012-05,svadz,c4ird5f,OPEN,wallstreetbets_comment_part1.csv,5,92934,2012-05-01,2012-05-01 15:44:24+00:00,matched
9,2012-05,svadz,c4irgzo,GET,wallstreetbets_comment_part1.csv,6,85426,2012-05-01,2012-05-01 15:53:02+00:00,matched


完成： c:\Users\eagle\RedditStock\Data\process\02_02_filtered_comment_ticker\2026-10-15_05-58-53_UTC\comment_ticker_cleaned.parquet


## 論文 Table 1：兩組 Reddit 樣本描述統計

沿用本次清理後資料及作者補回方式，僅在此統計段落區分 Full Sample（全部期間）與 2018–End Sample（留言 UTC 日期自 2018-07-01 起至資料末日）。先篩選留言，再分別重建 thread、author、comment 母體，不從全期間彙總推算子樣本。

Panel A：authors_per_thread、comments_per_thread、permnos_per_thread、permnos_per_author、permnos_per_comment，報告 `N, Mean, Std. Dev., Min, P10, Median, P90, Max`。std 採 ddof=1；P10/P90 採線性內插。原有 std/p90/max 欄位仍保留為相同數值的別名。

Panel B：thread、author、comment 各自提及至少兩個 distinct PERMNO 的比例，保留 numerator、denominator，分母與 Panel A 對應母體一致。相同 PERMNO 重複提及及 ticker 別名不增加股票數。

Comment 鍵維持 `(source_file, source_row)`。空白、[deleted]、[removed] author 不算可辨識作者；其留言仍保留。缺少 thread 的留言不進入 thread 母體，仍可計入 author/comment 母體。Full Sample 保留無日期留言並記錄數量；子樣本無法判斷日期者另行記錄，不猜測日期。

每個樣本記錄實際 min_date/max_date 與基本數量，輸出到新 statistics 目錄的 full_sample/、2018_end_sample/；根目錄的舊檔名維持 Full Sample 相容語義，另提供可直接整理 Table 1 的 table1_panel_a.csv、table1_panel_b.csv、table1_sample_summary.csv。既有清理流程、審核與舊輸出均不改動。


In [6]:
from pathlib import Path
from datetime import datetime, timezone
import json, time, shutil, hashlib
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

ACTIVITY_INPUT = None
ac_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'Data/process').is_dir())
if ACTIVITY_INPUT is not None:
    ac_source = Path(ACTIVITY_INPUT)
    if not ac_source.is_absolute(): ac_source = ac_root/ac_source
elif 'DEST' in globals():
    ac_source = Path(DEST)
else:
    ac_candidates = []
    for manifest in (ac_root/'Data/process/02_02_filtered_comment_ticker').glob('*/run_summary.json'):
        info = json.loads(manifest.read_text(encoding='utf-8'))
        source = manifest.parent/'comment_ticker_cleaned.parquet'
        if info.get('status') == 'complete' and source.is_file(): ac_candidates.append(source)
    if not ac_candidates: raise FileNotFoundError('找不到成功的 02_02 輸出')
    ac_source = max(ac_candidates, key=lambda p:p.parent.name)
ac_manifest = json.loads((ac_source.parent/'run_summary.json').read_text(encoding='utf-8'))
if ac_manifest.get('status') != 'complete': raise ValueError('02_02 尚未成功完成')
ac_schema = pq.read_schema(ac_source).names
if 'permno' not in ac_schema: raise ValueError('來源缺少 PERMNO；請執行新版 02_02')
ac_stat = (ac_source.stat().st_size, ac_source.stat().st_mtime_ns)
ac_started = time.perf_counter()
ac_columns = ['source_file','source_row','thread_id','permno']
ac_date_column = next((c for c in ['created_datetime_utc','created_date'] if c in ac_schema),None)
if ac_date_column is None: raise ValueError('缺少留言日期，無法建立 Table 1 日期樣本。')
ac_columns.append(ac_date_column)
if 'author' in ac_schema: ac_columns.append('author')
ac_mentions = pd.read_parquet(ac_source, columns=ac_columns)
if ac_mentions[['source_file','source_row','permno']].isna().any().any(): raise ValueError('缺少來源鍵或 PERMNO')
if not ac_mentions.permno.gt(0).all() or not ac_mentions.permno.mod(1).eq(0).all(): raise ValueError('PERMNO 必須為正整數')
ac_mentions['permno'] = ac_mentions.permno.astype('int64')
ac_mentions['thread_id'] = ac_mentions.thread_id.fillna('').str.strip()
ac_mentions['comment_date'] = pd.to_datetime(ac_mentions[ac_date_column],utc=True,errors='coerce').dt.normalize()
ac_comment_columns = ['source_file','source_row','thread_id','comment_date'] + (['author'] if 'author' in ac_columns else [])
ac_comments = ac_mentions[ac_comment_columns].drop_duplicates()
if ac_comments.duplicated(['source_file','source_row']).any(): raise ValueError('同一留言的 thread/author 不一致')
ac_author_sources = []
ac_author_input_stats = {}
if 'author' not in ac_comments:
    ac_matching_source = Path(ac_manifest['source'])
    if not ac_matching_source.is_absolute(): ac_matching_source = ac_root/ac_matching_source
    ac_matching_run = ac_matching_source.parent
    ac_parts = sorted(ac_matching_run.glob('*.comments.parquet'))
    if not ac_parts: raise FileNotFoundError('找不到來源 02 comments Parquet，無法補回 author')
    ac_keys = ac_comments[['source_file','source_row']]
    ac_recovered = []
    for part in ac_parts:
        ac_author_input_stats[str(part)] = (part.stat().st_size,part.stat().st_mtime_ns)
        source_file = part.name.removesuffix('.comments.parquet') + '.csv'
        wanted = ac_keys.loc[ac_keys.source_file.eq(source_file)]
        if wanted.empty: continue
        for batch in pq.ParquetFile(part).iter_batches(batch_size=100_000, columns=['source_file','source_row','author']):
            matched = batch.to_pandas().merge(wanted,on=['source_file','source_row'],how='inner',validate='one_to_one')
            if len(matched): ac_recovered.append(matched)
        ac_author_sources.append(str(part))
        print('author recovery:',part.name,flush=True)
    if not ac_recovered: raise ValueError('沒有留言成功取得 author')
    authors = pd.concat(ac_recovered,ignore_index=True)
    ac_comments = ac_comments.merge(authors,on=['source_file','source_row'],how='left',validate='one_to_one',indicator=True)
    if not ac_comments['_merge'].eq('both').all(): raise ValueError('清理後留言無法對應來源 author')
    ac_comments.drop(columns='_merge',inplace=True)
ac_comments['author'] = ac_comments.author.fillna('').str.strip()
ac_comments['known_author'] = ~ac_comments.author.str.lower().isin(['','[deleted]','[removed]'])
ac_mentions = ac_mentions.drop(columns='author',errors='ignore').merge(
    ac_comments[['source_file','source_row','author','known_author']],on=['source_file','source_row'],how='left',validate='many_to_one')
assert len(ac_mentions) == pq.read_metadata(ac_source).num_rows == ac_manifest['after']['rows']

def activity_distribution(label, values):
    a = np.asarray(values,dtype=np.int64)
    mean = float(a.mean()) if len(a) else None
    median = float(np.median(a)) if len(a) else None
    std = float(a.std(ddof=1)) if len(a)>1 else None
    p10 = float(np.percentile(a,10,method='linear')) if len(a) else None
    p90 = float(np.percentile(a,90,method='linear')) if len(a) else None
    minimum,maximum = (int(a.min()),int(a.max())) if len(a) else (None,None)
    # Keep legacy keys and add the exact Table 1 headings.
    return {'metric':label,'N':len(a),'Mean':mean,'Median':median,'std':std,'p90':p90,'max':maximum,
            'Std. Dev.':std,'Min':minimum,'P10':p10,'P90':p90,'Max':maximum}

def table1_sample(mentions, sample_name, requested_start=None):
    # Filter raw retained mention rows FIRST. Then rebuild all unit populations.
    subset = mentions if requested_start is None else mentions.loc[
        mentions.comment_date.ge(pd.Timestamp(requested_start,tz='UTC'))]
    comments = subset[['source_file','source_row','thread_id','comment_date','author','known_author']].drop_duplicates()
    if comments.duplicated(['source_file','source_row']).any(): raise ValueError('同一來源留言的日期或屬性不一致')
    valid_threads = comments.loc[comments.thread_id.ne('')]
    thread = valid_threads.groupby('thread_id').size().rename('n_comments').to_frame()
    thread['n_authors'] = valid_threads.loc[valid_threads.known_author].groupby('thread_id').author.nunique().reindex(thread.index,fill_value=0)
    thread['n_permnos'] = subset.loc[subset.thread_id.ne('')].groupby('thread_id').permno.nunique().reindex(thread.index,fill_value=0)
    thread = thread.reset_index()[['thread_id','n_authors','n_comments','n_permnos']]
    author = subset.loc[subset.known_author].groupby('author').permno.nunique().rename('n_permnos').reset_index()
    comment = subset.groupby(['source_file','source_row']).permno.nunique().rename('n_permnos').reset_index()
    assert int(thread.n_comments.sum()) == len(valid_threads)
    assert len(comment) == len(comments)
    dates = comments.comment_date.dropna()
    min_date = dates.min().date().isoformat() if len(dates) else None
    max_date = dates.max().date().isoformat() if len(dates) else None
    distribution = pd.DataFrame([
        activity_distribution('authors_per_thread',thread.n_authors),
        activity_distribution('comments_per_thread',thread.n_comments),
        activity_distribution('permnos_per_thread',thread.n_permnos),
        activity_distribution('permnos_per_author',author.n_permnos),
        activity_distribution('permnos_per_comment',comment.n_permnos),
    ])
    distribution['sample']=sample_name
    distribution['min_date']=min_date;distribution['max_date']=max_date
    comention=[]
    for unit,values in [('threads',thread.n_permnos),('authors',author.n_permnos),('comments',comment.n_permnos)]:
        numerator,denominator = int(values.ge(2).sum()),len(values)
        percentage = 100*numerator/denominator if denominator else None
        if percentage is not None: assert 0<=percentage<=100
        comention.append({'sample':sample_name,'metric':unit+'_mentioning_ge2_permnos_pct',
            'numerator':numerator,'denominator':denominator,'percentage':percentage,
            'min_date':min_date,'max_date':max_date})
    panel_b = pd.DataFrame(comention)
    for unit,metric in [('threads','permnos_per_thread'),('authors','permnos_per_author'),('comments','permnos_per_comment')]:
        assert panel_b.loc[panel_b.metric.eq(unit+'_mentioning_ge2_permnos_pct'),'denominator'].iloc[0] == distribution.loc[distribution.metric.eq(metric),'N'].iloc[0]
    if requested_start is not None and len(dates): assert min_date>=requested_start
    summary = {'status':'complete','sample':sample_name,'source':str(ac_source),
        'requested_start':requested_start,'min_date':min_date,'max_date':max_date,
        'mention_rows':len(subset),'unique_comments':len(comments),'unique_threads':len(thread),
        'unique_authors':len(author),'unique_permnos':int(subset.permno.nunique()),
        'comments_missing_thread':int(comments.thread_id.eq('').sum()),
        'comments_unknown_author':int((~comments.known_author).sum()),
        'comments_missing_date':int(comments.comment_date.isna().sum()),
        'comments_without_date_excluded_from_late_sample':int(ac_comments.comment_date.isna().sum()) if requested_start else 0,
        'comment_key':['source_file','source_row'],'permno_count':'distinct PERMNOs, not ticker mentions',
        'author_policy':'Exclude empty/[deleted]/[removed] identities only from author counts; retain their comments.',
        'population':'Date-filtered retained mentions; no other new cleaning or exclusion rules.',
        'std_ddof':1,'p90_method':'linear','p10_method':'linear','author_sources':ac_author_sources}
    summary['distribution'] = distribution.astype(object).where(distribution.notna(),None).to_dict(orient='records')
    summary['co_mention_statistics'] = panel_b.astype(object).where(panel_b.notna(),None).to_dict(orient='records')
    return distribution,panel_b,summary,thread,author,comment

ac_samples = {
    'full_sample':table1_sample(ac_mentions,'full_sample'),
    '2018_end_sample':table1_sample(ac_mentions,'2018_end_sample','2018-07-01'),
}
assert ac_stat == (ac_source.stat().st_size,ac_source.stat().st_mtime_ns)
for part,stat in ac_author_input_stats.items():
    assert stat == (Path(part).stat().st_size,Path(part).stat().st_mtime_ns)
ac_out = ac_source.parent/'statistics'/datetime.now(timezone.utc).strftime('%Y-%m-%d_%H-%M-%S_UTC')
ac_out.mkdir(parents=True,exist_ok=False)
for name,(distribution,panel_b,summary,thread,author,comment) in ac_samples.items():
    folder=ac_out/name;folder.mkdir()
    distribution.to_csv(folder/'activity_distributions.csv',index=False,encoding='utf-8-sig')
    panel_b.to_csv(folder/'co_mention_statistics.csv',index=False,encoding='utf-8-sig')
    thread.to_parquet(folder/'thread_statistics.parquet',index=False,compression='zstd')
    author.to_parquet(folder/'author_statistics.parquet',index=False,compression='zstd')
    comment.to_parquet(folder/'comment_statistics.parquet',index=False,compression='zstd')
    (folder/'activity_summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2,allow_nan=False),encoding='utf-8')
ac_distributions,ac_panel_b,ac_summary,ac_thread,ac_author,ac_comment=ac_samples['full_sample']
# Root-level legacy filenames continue to represent the Full Sample.
for filename in ['activity_distributions.csv','co_mention_statistics.csv','thread_statistics.parquet','author_statistics.parquet','comment_statistics.parquet']:
    shutil.copyfile(ac_out/'full_sample'/filename,ac_out/filename)
ac_summary=dict(ac_summary, samples={name:result[2] for name,result in ac_samples.items()},
                runtime_seconds=time.perf_counter()-ac_started,
                missing_date_policy='Full Sample retains undated comments; late sample cannot date-classify them and reports their count.')
(ac_out/'activity_summary.json').write_text(json.dumps(ac_summary,ensure_ascii=False,indent=2,allow_nan=False),encoding='utf-8')
ac_panel_a_all = pd.concat([result[0] for result in ac_samples.values()],ignore_index=True)
ac_panel_a_all[['sample','min_date','max_date','metric','N','Mean','Std. Dev.','Min','P10','Median','P90','Max']].to_csv(ac_out/'table1_panel_a.csv',index=False,encoding='utf-8-sig')
ac_panel_b_all = pd.concat([result[1] for result in ac_samples.values()],ignore_index=True)
ac_panel_b_all.to_csv(ac_out/'table1_panel_b.csv',index=False,encoding='utf-8-sig')
ac_basic_columns=['sample','requested_start','min_date','max_date','mention_rows','unique_comments','unique_threads','unique_authors','unique_permnos','comments_missing_thread','comments_unknown_author','comments_missing_date']
pd.DataFrame([{key:result[2][key] for key in ac_basic_columns} for result in ac_samples.values()]).to_csv(ac_out/'table1_sample_summary.csv',index=False,encoding='utf-8-sig')
display(ac_panel_a_all[['sample','metric','N','Mean','Std. Dev.','Min','P10','Median','P90','Max']])
display(ac_panel_b_all)
print('Table 1 統計輸出：',ac_out)


author recovery: wallstreetbets_comment_part1.comments.parquet
author recovery: wallstreetbets_comment_part2.comments.parquet
author recovery: wallstreetbets_comment_part3.comments.parquet
author recovery: wallstreetbets_comment_part4.comments.parquet
author recovery: wallstreetbets_comment_part5.comments.parquet
author recovery: wallstreetbets_comment_part6.comments.parquet


,sample,metric,N,Mean,Std. Dev.,Min,P10,Median,P90,Max
0,full_sample,authors_per_thread,221914,10.107357,75.523396,1,1.0,3.0,14.0,7863
1,full_sample,comments_per_thread,221914,16.866106,184.267728,1,1.0,3.0,17.0,11794
2,full_sample,permnos_per_thread,221914,8.114346,18.826029,1,1.0,4.0,17.0,571
3,full_sample,permnos_per_author,212119,9.643719,20.205398,1,1.0,3.0,25.0,689
4,full_sample,permnos_per_comment,3742825,1.334399,0.814928,1,1.0,1.0,2.0,308
5,2018_end_sample,authors_per_thread,132668,13.364202,97.301609,1,1.0,3.0,16.0,7863
6,2018_end_sample,comments_per_thread,132668,23.776382,237.885127,1,1.0,3.0,20.0,11794
7,2018_end_sample,permnos_per_thread,132668,9.176749,23.398040,1,1.0,4.0,19.0,571
8,2018_end_sample,permnos_per_author,179968,9.350101,19.669694,1,1.0,2.0,24.0,423
9,2018_end_sample,permnos_per_comment,3154365,1.313879,0.748317,1,1.0,1.0,2.0,94


,sample,metric,numerator,denominator,percentage,min_date,max_date
0,full_sample,threads_mentioning_ge2_permnos_pct,168634,221914,75.990699,2012-05-01,2020-07-31
1,full_sample,authors_mentioning_ge2_permnos_pct,138156,212119,65.131365,2012-05-01,2020-07-31
2,full_sample,comments_mentioning_ge2_permnos_pct,861525,3742825,23.018041,2012-05-01,2020-07-31
3,2018_end_sample,threads_mentioning_ge2_permnos_pct,98522,132668,74.262068,2018-07-01,2020-07-31
4,2018_end_sample,authors_mentioning_ge2_permnos_pct,115297,179968,64.065278,2018-07-01,2020-07-31
5,2018_end_sample,comments_mentioning_ge2_permnos_pct,696823,3154365,22.090754,2018-07-01,2020-07-31


Table 1 統計輸出： c:\Users\eagle\RedditStock\Data\process\02_02_filtered_comment_ticker\2026-10-15_05-58-53_UTC\statistics\2026-10-15_06-00-19_UTC
